# MACHO raw data

Author: Konstantin Malanchev

Last run: 2026-09-16

https://macho.nci.org.au/

One row per star for the whole survey, with a nested light curve. Sources:

- `star_view` (TAP): positions of 76.0M stars over 82 LMC, 6 SMC and 94 bulge fields
- `starstat_view` (TAP): per-star statistics, median magnitudes and microlensing search outputs
- photometry file archive: two-band light curves, 1.44 TiB over 182 fields

Magnitudes are instrumental. We add Kron-Cousins `kv` / `kr` from the MACHO calibration of
Alcock et al. 1999 ([1999PASP..111.1539A](https://ui.adsabs.harvard.edu/abs/1999PASP..111.1539A)).

In [1]:
import gzip
import re
import shutil
from types import SimpleNamespace
from urllib.parse import urlencode
from urllib.request import urlopen

import nested_pandas as npd
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute
import pyarrow.csv
import pyarrow.parquet
from dask.distributed import Client, as_completed
from hats_import.pipeline import _send_failure_email, _send_success_email
from nested_pandas import NestedFrame
from tqdm.auto import tqdm
from upath import UPath

/astro/users/kmalanch/.virtualenvs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dir = UPath("/astro/store/shire/hats/raw/macho")
star_raw_dir = UPath("/astro/store/shire/hats/raw/macho_star")
lc_parquet_dir = UPath("/astro/store/shire/hats/raw/macho_lc_parquet")
parquet_dir = UPath("/astro/store/shire/hats/raw/macho_parquet")

photometry_url = UPath("https://macho.nci.org.au/macho_photometry")
tap_url = "https://machotap.asvo.nci.org.au/ncitap/tap/sync"

sentinels = [-99.0, -9999.0]
standard_exposure = {"LMC": 300.0, "SMC": 600.0, "Bulge": 150.0}

# Alcock+99 (1999PASP..111.1539A) eq (1) & (2) with chunk offset 0 and template
# airmass 1, which is what MACHO's own varstar_view.magave_kv / magave_kr use.
calibration_zeropoint = {"V": 24.32 - 2.5 * np.log10(300.0), "R": 24.06 - 2.5 * np.log10(300.0)}
calibration_color = -0.180

for directory in [raw_dir, star_raw_dir, lc_parquet_dir, parquet_dir]:
    directory.mkdir(parents=True, exist_ok=True)


def field_region(field):
    if field <= 82:
        return "LMC"
    if 206 <= field <= 213:
        return "SMC"
    return "Bulge"

In [3]:
email_args = SimpleNamespace(completion_email_address="kmalanch@andrew.cmu.edu", output_artifact_name="macho_raw")


# Email when any later cell fails, the last cell emails on success.
def email_on_failure(result):
    if not result.success:
        _send_failure_email(email_args, result.error_in_exec or result.error_before_exec)


events = get_ipython().events
if all(callback.__name__ != "email_on_failure" for callback in events.callbacks["post_run_cell"]):
    events.register("post_run_cell", email_on_failure)

## Photometry

In [4]:
# iterdir() also yields the index page's column-sort links, keep the data entries only.
field_dirs = sorted(
    (path for path in photometry_url.iterdir() if re.fullmatch(r"F_\d+", path.name)),
    key=lambda path: int(path.name[2:]),
)
fields = [int(path.name[2:]) for path in field_dirs]
len(fields)

182

In [5]:
def list_tiles(field_dir):
    return [path for path in field_dir.iterdir() if path.suffix == ".gz"]


def download(remote_path):
    path = raw_dir / remote_path.name
    if path.exists():
        return path
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with remote_path.open("rb") as remote_file, tmp_path.open("wb") as file:
        shutil.copyfileobj(remote_file, file)
    tmp_path.rename(path)
    return path


with Client(n_workers=4, threads_per_worker=1) as client:
    futures = client.map(list_tiles, field_dirs)
    remote_paths = [
        path for future in tqdm(as_completed(futures), total=len(futures), desc="list") for path in future.result()
    ]
    futures = client.map(download, remote_paths)
    for future in tqdm(as_completed(futures), total=len(futures), desc="download"):
        future.result()
len(remote_paths)

/astro/users/kmalanch/.virtualenvs/default/lib/python3.12/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 32865 instead
  warnings.warn(
download: 100%|██████████| 29022/29022 [00:26<00:00, 1080.84it/s]


29022

## Stars

`star_view` for positions, `starstat_view` for statistics

In [6]:
def query(adql):
    return f"{tap_url}?" + urlencode({"REQUEST": "doQuery", "LANG": "ADQL", "FORMAT": "csv", "QUERY": adql})


def download_csv(path, adql):
    if path.exists():
        return path
    tmp_path = path.with_suffix(".gz.tmp")
    with urlopen(query(adql), timeout=3600) as response, gzip.open(tmp_path, "wb") as file:
        shutil.copyfileobj(response, file)
    tmp_path.rename(path)
    return path


for field in tqdm(fields):
    download_csv(
        star_raw_dir / f"star_{field}.csv.gz",
        f"SELECT starid, field, tile, seqn, rarad, decrad FROM public.star_view WHERE field = {field}",
    )
    download_csv(
        star_raw_dir / f"starstat_{field}.csv.gz",
        f"SELECT * FROM public.starstat_view WHERE fieldid = {field}",
    )

100%|██████████| 182/182 [00:00<00:00, 1343.71it/s]


## Convert tiles to nested light curves


In [7]:
band_columns = [
    "mag", "err", "normsky", "type", "crowd", "chi2", "mpix", "cosmicrf",
    "amp", "xpix", "ypix", "avesky", "fwhm", "tobs", "cut",
]
phot_columns = [
    "_empty", "fieldid", "tileid", "seqn", "dateobs", "obsid", "sideofpier", "exposure", "airmass",
] + [f"{band}{name}" for band in "rb" for name in band_columns]

# Columns that hold a measured value, i.e. the ones that carry the sentinels.
value_columns = ["mag", "err", "normsky", "type", "crowd", "chi2", "mpix", "cosmicrf", "amp", "avesky", "fwhm"]

drop_columns = ["_empty", "fieldid", "rxpix", "rypix", "bxpix", "bypix"]
keep_columns = [column for column in phot_columns if column not in drop_columns]

pier_type = pa.dictionary(pa.int8(), pa.string())
lc_types = {
    "tileid": pa.int32(),
    "seqn": pa.int32(),
    "dateobs": pa.float64(),
    "obsid": pa.int32(),
    "sideofpier": pa.string(),
    "exposure": pa.float32(),
    "airmass": pa.float32(),
    **{f"{band}{name}": pa.float32() for band in "rb" for name in value_columns},
    **{f"{band}tobs": pa.int32() for band in "rb"},
    **{f"{band}cut": pa.int8() for band in "rb"},
}

In [8]:
def null_out(table, column, mask):
    values = table[column]
    return table.set_column(
        table.schema.get_field_index(column),
        column,
        pa.compute.if_else(pa.compute.fill_null(mask, True), pa.scalar(None, values.type), values),
    )


def skip_invalid_row(row):
    # A few source tiles end with a truncated last line.
    print(f"skipping malformed row: {row.text[:60]}")
    return "skip"


def convert_tile(path):
    out_path = lc_parquet_dir / f"{path.stem}.parquet"
    if out_path.exists():
        return out_path

    table = pa.csv.read_csv(
        path,
        read_options=pa.csv.ReadOptions(column_names=phot_columns),
        # some of the input data files are truncated, we skip invalid rows
        parse_options=pa.csv.ParseOptions(delimiter=";", invalid_row_handler=skip_invalid_row),
        convert_options=pa.csv.ConvertOptions(include_columns=keep_columns, column_types=lc_types),
    )
    table = table.filter(pa.compute.or_(pa.compute.equal(table["rcut"], 1), pa.compute.equal(table["bcut"], 1)))

    for band in "rb":
        good = pa.compute.equal(table[f"{band}cut"], 1)
        for name in value_columns:
            column = f"{band}{name}"
            # -99 marks a missing measurement, 999 does the same in the quality columns.
            sentinel = pa.compute.is_in(table[column], value_set=pa.array([-99.0, 999.0], pa.float32()))
            table = null_out(table, column, pa.compute.or_(pa.compute.invert(good), sentinel))
        # A few percent of otherwise good rows carry a nonsense error, e.g. -9885.
        table = null_out(table, f"{band}err", pa.compute.less_equal(table[f"{band}err"], 0))
        table = table.set_column(table.schema.get_field_index(f"{band}cut"), f"{band}cut", good)
    table = table.set_column(
        table.schema.get_field_index("sideofpier"), "sideofpier", table["sideofpier"].cast(pier_type)
    )

    nf = NestedFrame.from_flat(npd.from_pyarrow(table), base_columns=["tileid"], on="seqn", name="lc").reset_index()
    tmp_path = out_path.with_suffix(".parquet.tmp")
    nf.to_parquet(tmp_path)
    tmp_path.rename(out_path)
    return out_path

In [9]:
# Tiles with no photometry at all are 29-byte files.
tile_files = sorted(path for path in raw_dir.glob("F_*.gz") if path.stat().st_size > 1000)

with Client(n_workers=16, threads_per_worker=1, memory_limit="8GB") as client:
    futures = client.map(convert_tile, tile_files)
    for future in tqdm(as_completed(futures), total=len(futures), desc="tiles"):
        future.result()

/astro/users/kmalanch/.virtualenvs/default/lib/python3.12/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43053 instead
  warnings.warn(
tiles: 100%|██████████| 25662/25662 [00:42<00:00, 604.32it/s] 


## Merge stars with light curves

`kv` / `kr` are Kron-Cousins V / R from the median instrumental magnitudes, using
eq (1) & (2) of Alcock et al. 1999, PASP 111, 1539
([1999PASP..111.1539A](https://ui.adsabs.harvard.edu/abs/1999PASP..111.1539A)).
The light curve stays instrumental, the published transformation is per star.

In [10]:
def calibrate(blue, red, exposure):
    common = calibration_color * (blue - red) + 2.5 * np.log10(exposure)
    return blue + calibration_zeropoint["V"] + common, red + calibration_zeropoint["R"] + common

In [11]:
tap_schema = pd.read_csv(query(
    "SELECT column_name, datatype FROM TAP_SCHEMA.columns "
    "WHERE table_name IN ('public.star_view', 'public.starstat_view')"
))
# Integer and floating columns as declared by the service, floats narrowed to float32.
star_types = {
    row.column_name: pa.int32() if row.datatype == "integer" else pa.float32()
    for row in tap_schema.itertuples()
} | {"rarad": pa.float64(), "decrad": pa.float64()}


def read_star_csv(path):
    table = pa.csv.read_csv(path, convert_options=pa.csv.ConvertOptions(column_types=star_types))
    for field in table.schema:
        if pa.types.is_floating(field.type):
            sentinel = pa.compute.is_in(table[field.name], value_set=pa.array(sentinels, field.type))
            table = null_out(table, field.name, sentinel)
    return table


def convert_field(field):
    out_dir = parquet_dir / f"macho_{field}"
    if out_dir.exists():
        return out_dir

    stats = read_star_csv(star_raw_dir / f"starstat_{field}.csv.gz").drop_columns(["fieldid", "tileid", "seqn"])
    stars = read_star_csv(star_raw_dir / f"star_{field}.csv.gz").join(stats, keys="starid", join_type="left outer")
    stars = npd.from_pyarrow(stars).rename(columns={"field": "fieldid", "tile": "tileid"})

    stars["ra"] = np.degrees(stars.pop("rarad").to_numpy(dtype="float64")) % 360.0
    stars["dec"] = np.degrees(stars.pop("decrad").to_numpy(dtype="float64"))
    stars["kv"], stars["kr"] = calibrate(
        stars["bmagave"], stars["rmagave"], standard_exposure[field_region(field)]
    )
    stars = stars.astype(
        {
            "ra": pd.ArrowDtype(pa.float64()),
            "dec": pd.ArrowDtype(pa.float64()),
            "kv": pd.ArrowDtype(pa.float32()),
            "kr": pd.ArrowDtype(pa.float32()),
        }
    )

    star_table = pa.table(stars).replace_schema_metadata(None)

    # One file per tile: a whole field does not fit in memory, and a nested column
    # cannot be read back when it spans more than one row group.
    tmp_dir = out_dir.with_suffix(".tmp")
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tile_files = sorted(lc_parquet_dir.glob(f"F_{field}.*.parquet"))
    tile_ids = [int(path.stem.split(".")[1]) for path in tile_files]
    lc_type = None
    for tile_path, tile_id in zip(tile_files, tile_ids):
        lc = pa.parquet.read_table(tile_path, columns=["seqn", "lc"])
        lc_type = lc.schema.field("lc").type
        chunk = star_table.filter(pa.compute.equal(star_table["tileid"], tile_id))
        rows = pa.compute.index_in(chunk["seqn"], value_set=lc["seqn"].combine_chunks())
        chunk = chunk.append_column("lc", lc["lc"].take(rows))
        pa.parquet.write_table(chunk.combine_chunks(), tmp_dir / f"{tile_id}.parquet")

    # Stars of tiles with no photometry file at all.
    leftover = star_table.filter(
        pa.compute.invert(pa.compute.is_in(star_table["tileid"], value_set=pa.array(tile_ids)))
    )
    if leftover.num_rows:
        leftover = leftover.append_column("lc", pa.nulls(leftover.num_rows, lc_type))
        pa.parquet.write_table(leftover.combine_chunks(), tmp_dir / "no_photometry.parquet")

    tmp_dir.rename(out_dir)
    return out_dir

In [12]:
with Client(n_workers=8, threads_per_worker=1, memory_limit="16GB") as client:
    futures = client.map(convert_field, fields)
    for future in tqdm(as_completed(futures), total=len(futures), desc="fields"):
        future.result()

/astro/users/kmalanch/.virtualenvs/default/lib/python3.12/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34335 instead
  warnings.warn(
fields: 100%|██████████| 182/182 [49:13<00:00, 16.23s/it] 


In [13]:
parquet_files = sorted(parquet_dir.glob("macho_*/*.parquet"))
total_rows = sum(pa.parquet.read_metadata(path).num_rows for path in parquet_files)
len(parquet_files), total_rows

(25667, 69173447)

In [14]:
# Should reproduce MACHO's published magave_kv / magave_kr to ~0.002 mag because of the rounding errors
varstar = pd.read_csv(query("SELECT magave_b, magave_r, magave_kv, magave_kr FROM public.varstar_view"))
varstar = varstar[(varstar["magave_b"] > -90) & (varstar["magave_r"] > -90)]
kv, kr = calibrate(varstar["magave_b"], varstar["magave_r"], standard_exposure["LMC"])
(kv - varstar["magave_kv"]).abs().max(), (kr - varstar["magave_kr"]).abs().max()

(np.float64(0.0018999999999991246), np.float64(0.0021799999999991826))

In [15]:
_send_success_email(email_args)